# SQL practice

*Module 3 Part 1. 21 exercises against a real database.*

Everything here runs against **`data/bikeshare.db`**, a real SQLite
database of 19,478 rows. No account, no server, no signup, and nothing
to install — the file is already beside this notebook.

## What is in the database

| Table | Rows | One row is |
|---|---|---|
| `rides_daily` | 731 | one day, 2011–2012 |
| `rides_hourly` | 17,379 | one hour |
| `stations` | 858 | one Capital Bikeshare station |
| `housing` | 506 | one Boston town |
| `weather` | 4 | a weather code and its meaning |
| `season` | 4 | a season code |
| `year_lookup` | 2 | 0 → 2011, 1 → 2012 |
| `daily_summary` | 731 | a **view** joining the three lookups |

All of it is real. The ride data is the UCI Bike Sharing Dataset
(Capital Bikeshare, Washington D.C.); the station list is a real
Capital Bikeshare GBFS feed; the weather and season descriptions are
transcribed from the dataset's own readme.

**Two facts about this data are worth knowing before you start**, and
both come up in the exercises:

- **54 of the 858 stations have no `region_id`.** They are NULL in the
  source feed. That is real, not a mistake.
- **Weather code 4 (heavy rain or snow) never appears in
  `rides_daily`**, though it exists in `weather` and does appear in
  `rides_hourly`. Exercise 11 is built on this.

## How to open the database

**Four ways, all free, none needing an account.** Pick one.

| | How |
|---|---|
| **This notebook** | Just run the next cell. Easiest. |
| **DB Browser for SQLite** | Free desktop app, <https://sqlitebrowser.org/> — open `data/bikeshare.db`, use the *Execute SQL* tab |
| **VS Code** | Install the "SQLite Viewer" or "SQLite" extension, then click the `.db` file |
| **Browser, nothing installed** | <https://sqliteonline.com/> → *File* → *Open DB* → choose `data/bikeshare.db` |

You do **not** need to install SQLite from sqlite.org, and you do not
need a Mode or ThoughtSpot account. If you want to work through the
ThoughtSpot tutorial as well, it is good and it is free to read:
<https://www.thoughtspot.com/sql-tutorial/introduction-to-sql>

## How to use this workbook

Each exercise has a cell like this:

```python
answer_1 = """
    -- your SQL here
"""

check(1, answer_1)
```

Write your SQL between the quotes and run the cell. `check()` runs it and
tells you whether the result is right — **it does not show you the
answer.** Full solutions are in the answer key at the very bottom, for
after you have tried.

## Setup — run this first

In [ ]:
import hashlib
import os
import sqlite3
import textwrap

import pandas as pd


def find_database():
    """The .db lives in data/ - look there and here."""
    for candidate in ("bikeshare.db", os.path.join("data", "bikeshare.db")):
        if os.path.exists(candidate):
            return candidate
    raise FileNotFoundError(
        "bikeshare.db not found. It should be in the data/ folder "
        "beside this notebook."
    )


DATABASE = find_database()
db = sqlite3.connect(DATABASE)

print("connected to", DATABASE)
print()

tables = pd.read_sql_query("""
    SELECT name, type
    FROM sqlite_master
    WHERE type IN ('table', 'view')
    ORDER BY type, name
""", db)

for _, row in tables.iterrows():
    n = pd.read_sql_query("SELECT COUNT(*) AS n FROM " + row["name"],
                          db)["n"][0]
    print("  {:<14} {:<6} {:>6,} rows".format(row["name"], row["type"], n))

In [ ]:
def q(sql):
    """Run any SQL and show the result. Use this to explore."""
    return pd.read_sql_query(sql, db)


# The self-marking machinery. You never need to read this - but it is
# here rather than hidden, because nothing in this course is magic.
def _canonical(rows, columns):
    lines = [str(len(columns))]
    for row in rows:
        cells = []
        for value in row:
            if value is None:
                cells.append("<NULL>")
            elif isinstance(value, float):
                cells.append("{:.6f}".format(round(value, 6)))
            else:
                cells.append(str(value))
        lines.append("\x1f".join(cells))
    return "\x1e".join(lines)


def _fingerprint(rows, columns):
    return hashlib.sha256(_canonical(rows, columns).encode()).hexdigest()[:16]


def check(number, sql):
    """Run your SQL and say whether it is right."""
    expected = EXPECTED[number]

    if sql is None or "your SQL here" in sql or not sql.strip():
        print("Exercise {}: nothing to check yet - write some SQL.".format(number))
        return

    try:
        cursor = db.execute(sql)
        columns = [d[0] for d in cursor.description]
        rows = cursor.fetchall()
    except Exception as problem:
        print("Exercise {}: SQL did not run.".format(number))
        print("   {}: {}".format(type(problem).__name__, problem))
        return

    if _fingerprint(rows, columns) == expected["fingerprint"]:
        print("Exercise {}: CORRECT  ({} rows, {} columns)".format(
            number, len(rows), len(columns)))
        return

    print("Exercise {}: not right yet.".format(number))
    if len(columns) != expected["n_columns"]:
        print("   Expected {} columns, got {}.".format(
            expected["n_columns"], len(columns)))
    if len(rows) != expected["n_rows"]:
        print("   Expected {} rows, got {}.".format(
            expected["n_rows"], len(rows)))
    if (len(rows) == expected["n_rows"]
            and len(columns) == expected["n_columns"]):
        print("   Right shape, wrong contents - check the values, the "
              "column order,")
        print("   and the row order (ORDER BY matters here).")
    print()
    print("   Your result:")
    frame = pd.DataFrame(rows, columns=columns)
    print(textwrap.indent(frame.head(6).to_string(), "     "))


print("Ready. Explore with q('SELECT ...'), check answers with check(n, sql).")

In [ ]:
# The expected results. These are SHA-256 fingerprints, not answers -
# reading them will not help you, but check() needs them.
EXPECTED = {
    1: dict(fingerprint="bb419bc08819aac0", n_rows=5, n_columns=2),
    2: dict(fingerprint="ec207ad995252714", n_rows=12, n_columns=3),
    3: dict(fingerprint="91e4344351b31e45", n_rows=1, n_columns=1),
    4: dict(fingerprint="d6411b4ad8151abc", n_rows=54, n_columns=2),
    5: dict(fingerprint="8e647bd81c0e3b45", n_rows=4, n_columns=4),
    6: dict(fingerprint="72e469dc43f0edd1", n_rows=9, n_columns=3),
    7: dict(fingerprint="cec5d7372e1aa872", n_rows=4, n_columns=2),
    8: dict(fingerprint="63ef8eea60fb50cf", n_rows=2, n_columns=3),
    9: dict(fingerprint="be4f94495f4e27cc", n_rows=1, n_columns=3),
    10: dict(fingerprint="d52a2516f117784c", n_rows=3, n_columns=2),
    11: dict(fingerprint="2c8948e4b4f912ed", n_rows=4, n_columns=3),
    12: dict(fingerprint="d104d549c874a698", n_rows=3, n_columns=4),
    13: dict(fingerprint="f591a7ec169c8d6f", n_rows=24, n_columns=3),
    14: dict(fingerprint="dc71271949e7ab41", n_rows=1, n_columns=3),
    15: dict(fingerprint="ba9f6ee449e0d26b", n_rows=10, n_columns=3),
    16: dict(fingerprint="230276796c40d6b6", n_rows=4, n_columns=3),
    17: dict(fingerprint="bb419bc08819aac0", n_rows=5, n_columns=2),
    18: dict(fingerprint="f72112eec83f6614", n_rows=12, n_columns=3),
    19: dict(fingerprint="2a3323857cf4f132", n_rows=24, n_columns=3),
    20: dict(fingerprint="778aa51982194bdb", n_rows=1, n_columns=4),
    21: dict(fingerprint="250eb70cf4b5eae5", n_rows=5, n_columns=4),
}

print("loaded expected results for", len(EXPECTED), "exercises")

### Warm-up — not marked

Run these to get a feel for the tables before the exercises start.

In [ ]:
# Every table's shape, and the first few rows of the main one.
print(q("SELECT * FROM weather"))
print()
print(q("SELECT * FROM season"))
print()
print(q("SELECT dteday, season_id, weather_id, casual, registered, cnt "
        "FROM rides_daily LIMIT 5"))

In [ ]:
# The view does the lookup joins for you - compare with the raw table.
print(q("SELECT * FROM daily_summary LIMIT 5"))

### The schema, whenever you need it

Run this any time you forget a column name.

In [ ]:
for _, row in q("""SELECT name, sql FROM sqlite_master
                   WHERE type = 'table' ORDER BY name""").iterrows():
    print("=" * 66)
    print(row["sql"])

---

## Level 1 - Reading one table

### Exercise 1

Return the `dteday` and `cnt` of the 5 busiest days, busiest first.

*Hint: ORDER BY ... DESC, then LIMIT.*

In [ ]:
answer_1 = """
    -- your SQL here
"""

check(1, answer_1)

### Exercise 2

Return `dteday`, `cnt` and `temp` for every day in 2012 (`year_id = 1`) where more than 8000 rides were taken. Order by `dteday`.

*Hint: Two conditions joined with AND.*

In [ ]:
answer_2 = """
    -- your SQL here
"""

check(2, answer_2)

### Exercise 3

How many stations have a capacity of 20 or more? Return a single column named `n`. Read the wording carefully — 22 stations have a capacity of exactly 20, so `>` and `>=` give different answers here.

*Hint: COUNT(*) with a WHERE. 'Or more' includes 20 itself.*

In [ ]:
answer_3 = """
    -- your SQL here
"""

check(3, answer_3)

### Exercise 4

Return `station_id` and `name` for every station whose `region_id` is missing. Order by `name`.

*Hint: NULL is not a value. `= NULL` never matches anything.*

In [ ]:
answer_4 = """
    -- your SQL here
"""

check(4, answer_4)

---

## Level 2 - Aggregation and grouping

### Exercise 5

For each `season_id`, return the number of days (`days`), total rides (`total`) and average rides rounded to 1 decimal place (`average`). Order by `season_id`.

*Hint: GROUP BY, and ROUND(AVG(...), 1).*

In [ ]:
answer_5 = """
    -- your SQL here
"""

check(5, answer_5)

### Exercise 6

For each `region_id` in `stations`, return the number of stations (`n`) and total capacity (`total_capacity`). Include the missing-region group. Order by `region_id`, nulls first.

*Hint: GROUP BY puts all NULLs in one group. SQLite sorts NULL first by default in ASC.*

In [ ]:
answer_6 = """
    -- your SQL here
"""

check(6, answer_6)

### Exercise 7

Return each `mnth` in 2012 (`year_id = 1`) whose total rides exceed 200000, as `mnth` and `total`. Order by `total` descending.

*Hint: A condition on rows and a condition on groups are different clauses.*

In [ ]:
answer_7 = """
    -- your SQL here
"""

check(7, answer_7)

### Exercise 8

In `housing`, return the number of towns (`n`) and the average `MEDV` rounded to 3 decimals (`avg_medv`) for towns beside the Charles River (`CHAS = 1`) and those not, as columns `CHAS`, `n`, `avg_medv`. Order by `CHAS`.

*Hint: GROUP BY a column that only has two values.*

In [ ]:
answer_8 = """
    -- your SQL here
"""

check(8, answer_8)

### Exercise 9

Return the busiest single hour of the whole dataset: `dteday`, `hr` and `cnt`, one row.

*Hint: Order and limit, or a subquery with MAX.*

In [ ]:
answer_9 = """
    -- your SQL here
"""

check(9, answer_9)

---

## Level 3 - Joins

### Exercise 10

Return `label` (from `weather`) and the total rides for each weather type in `rides_daily`, as `label` and `total`. Use an INNER JOIN. Order by `total` descending.

*Hint: Join rides_daily to weather on weather_id.*

In [ ]:
answer_10 = """
    -- your SQL here
"""

check(10, answer_10)

### Exercise 11

Now list EVERY weather type, including any that never occurs in `rides_daily`, as `weather_id`, `label` and `days` (0 where it never occurs). Order by `weather_id`. This is the exercise that shows why the join type matters.

*Hint: Start FROM weather, not from rides_daily. COUNT(*) would count the unmatched row as 1 - count the other table's column instead.*

In [ ]:
answer_11 = """
    -- your SQL here
"""

check(11, answer_11)

### Exercise 12

Return `dteday`, `season` label, `weather` label and `cnt` for the 3 busiest days, busiest first. Columns: `dteday`, `season`, `weather`, `cnt`.

*Hint: Two joins, one to each lookup table.*

In [ ]:
answer_12 = """
    -- your SQL here
"""

check(12, answer_12)

### Exercise 13

For 2011-01-01 only, return `hr`, the hourly `cnt`, and that day's daily total as `day_total`. Columns: `hr`, `cnt`, `day_total`. Order by `hr`.

*Hint: Join the hourly table to the daily table on dteday.*

In [ ]:
answer_13 = """
    -- your SQL here
"""

check(13, answer_13)

### Exercise 14

Prove the two ride tables agree. Return one row with `daily_total`, `hourly_total` and `difference` (daily minus hourly).

*Hint: Two scalar subqueries in one SELECT.*

In [ ]:
answer_14 = """
    -- your SQL here
"""

check(14, answer_14)

---

## Level 4 - Window functions and subqueries

### Exercise 15

For the first 10 days of 2011 by date, return `dteday`, `cnt`, and a day-by-day running total as `running_total`. Use ROWS, not the default frame.

*Hint: SUM(...) OVER (ORDER BY ... ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW).*

In [ ]:
answer_15 = """
    -- your SQL here
"""

check(15, answer_15)

### Exercise 16

Rank the 4 seasons by total rides, most first. Return `season_id`, `total` and `rank` using RANK().

*Hint: A window function can sit on top of an aggregate.*

In [ ]:
answer_16 = """
    -- your SQL here
"""

check(16, answer_16)

### Exercise 17

Return every day whose `cnt` is above the overall daily average, as `dteday` and `cnt`, busiest first, limited to 5 rows.

*Hint: A subquery in the WHERE clause computes the average.*

In [ ]:
answer_17 = """
    -- your SQL here
"""

check(17, answer_17)

### Exercise 18

For each month of 2012, return `mnth`, that month's total as `total`, and the previous month's total as `prev_month` (NULL for January). Order by `mnth`.

*Hint: LAG() is a window function. LAG(x) OVER (ORDER BY ...).*

In [ ]:
answer_18 = """
    -- your SQL here
"""

check(18, answer_18)

---

## Level 5 - Challenge

### Exercise 19

For each hour of the day (0-23), return `hr`, the average rides on working days as `working`, and on non-working days as `non_working`, both rounded to 1 decimal. Order by `hr`. Do it in ONE query without a join.

*Hint: An aggregate can wrap a CASE expression: AVG(CASE WHEN ... THEN cnt END). CASE with no ELSE yields NULL, and AVG skips NULLs.*

In [ ]:
answer_19 = """
    -- your SQL here
"""

check(19, answer_19)

### Exercise 20

Find the single worst day for casual riders as a share of all riders, among days with at least 1000 total rides. Return `dteday`, `casual`, `cnt` and `casual_share` rounded to 4 decimals. One row.

*Hint: Integer division truncates. Multiply by 1.0 to force a real division.*

In [ ]:
answer_20 = """
    -- your SQL here
"""

check(20, answer_20)

### Exercise 21

Return the 5 stations furthest north (largest `lat`), with `name`, `capacity`, `lat`, and their capacity rank across ALL stations as `capacity_rank` (largest capacity = rank 1, using DENSE_RANK). Order by `lat` descending.

*Hint: Compute the rank over all rows first, in a subquery or CTE, then filter and order.*

In [ ]:
answer_21 = """
    -- your SQL here
"""

check(21, answer_21)

---

## Where to go next

- **Notebook 14**, `14-sql-and-databases.ipynb`, is the lecture material
  behind all of this — ACID, indexes, all five join types, window frames,
  and SQL injection.
- **The ThoughtSpot SQL tutorial** (slides 24 and 32) —
  <https://www.thoughtspot.com/sql-tutorial/introduction-to-sql>
- **Use The Index, Luke** — <https://use-the-index-luke.com/> — free, and
  the clearest explanation of indexes anywhere.
- **SQLite's own docs** — <https://www.sqlite.org/lang.html> is the
  complete syntax reference, and it is short.

### Try breaking things

The database is a file. Copy it, wreck the copy, learn something:

```python
import shutil
shutil.copy("data/bikeshare.db", "my-sandbox.db")
```

Then open `my-sandbox.db` instead and try `UPDATE` without a `WHERE`,
`DELETE FROM rides_daily`, inserting a row that violates a `CHECK`, or
adding a foreign key that points nowhere. You cannot hurt anything, and
the error messages are the lesson.

---

## Answer key

**Try every exercise before reading this.** A wrong query you then fix
teaches more than a right query you copied.

Where more than one query is correct, this is only *an* answer, not
*the* answer. If `check()` passed, you were right, whatever you wrote.


**Exercise 1** — Return the `dteday` and `cnt` of the 5 busiest days, busiest first.

```sql
SELECT dteday, cnt
FROM rides_daily
ORDER BY cnt DESC
LIMIT 5
```


**Exercise 2** — Return `dteday`, `cnt` and `temp` for every day in 2012 (`year_id = 1`) where more than 8000 rides were taken. Order by `dteday`.

```sql
SELECT dteday, cnt, temp
FROM rides_daily
WHERE year_id = 1 AND cnt > 8000
ORDER BY dteday
```


**Exercise 3** — How many stations have a capacity of 20 or more? Return a single column named `n`. Read the wording carefully — 22 stations have a capacity of exactly 20, so `>` and `>=` give different answers here.

```sql
SELECT COUNT(*) AS n
FROM stations
WHERE capacity >= 20
```


**Exercise 4** — Return `station_id` and `name` for every station whose `region_id` is missing. Order by `name`.

```sql
SELECT station_id, name
FROM stations
WHERE region_id IS NULL
ORDER BY name
```


**Exercise 5** — For each `season_id`, return the number of days (`days`), total rides (`total`) and average rides rounded to 1 decimal place (`average`). Order by `season_id`.

```sql
SELECT season_id,
       COUNT(*)           AS days,
       SUM(cnt)           AS total,
       ROUND(AVG(cnt), 1) AS average
FROM rides_daily
GROUP BY season_id
ORDER BY season_id
```


**Exercise 6** — For each `region_id` in `stations`, return the number of stations (`n`) and total capacity (`total_capacity`). Include the missing-region group. Order by `region_id`, nulls first.

```sql
SELECT region_id,
       COUNT(*)       AS n,
       SUM(capacity)  AS total_capacity
FROM stations
GROUP BY region_id
ORDER BY region_id
```


**Exercise 7** — Return each `mnth` in 2012 (`year_id = 1`) whose total rides exceed 200000, as `mnth` and `total`. Order by `total` descending.

```sql
SELECT mnth, SUM(cnt) AS total
FROM rides_daily
WHERE year_id = 1
GROUP BY mnth
HAVING SUM(cnt) > 200000
ORDER BY total DESC
```


**Exercise 8** — In `housing`, return the number of towns (`n`) and the average `MEDV` rounded to 3 decimals (`avg_medv`) for towns beside the Charles River (`CHAS = 1`) and those not, as columns `CHAS`, `n`, `avg_medv`. Order by `CHAS`.

```sql
SELECT CHAS,
       COUNT(*)            AS n,
       ROUND(AVG(MEDV), 3) AS avg_medv
FROM housing
GROUP BY CHAS
ORDER BY CHAS
```


**Exercise 9** — Return the busiest single hour of the whole dataset: `dteday`, `hr` and `cnt`, one row.

```sql
SELECT dteday, hr, cnt
FROM rides_hourly
ORDER BY cnt DESC
LIMIT 1
```


**Exercise 10** — Return `label` (from `weather`) and the total rides for each weather type in `rides_daily`, as `label` and `total`. Use an INNER JOIN. Order by `total` descending.

```sql
SELECT w.label, SUM(d.cnt) AS total
FROM rides_daily d
INNER JOIN weather w ON d.weather_id = w.weather_id
GROUP BY w.label
ORDER BY total DESC
```


**Exercise 11** — Now list EVERY weather type, including any that never occurs in `rides_daily`, as `weather_id`, `label` and `days` (0 where it never occurs). Order by `weather_id`. This is the exercise that shows why the join type matters.

```sql
SELECT w.weather_id,
       w.label,
       COUNT(d.dteday) AS days
FROM weather w
LEFT JOIN rides_daily d ON w.weather_id = d.weather_id
GROUP BY w.weather_id, w.label
ORDER BY w.weather_id
```


**Exercise 12** — Return `dteday`, `season` label, `weather` label and `cnt` for the 3 busiest days, busiest first. Columns: `dteday`, `season`, `weather`, `cnt`.

```sql
SELECT d.dteday,
       s.label AS season,
       w.label AS weather,
       d.cnt
FROM rides_daily d
JOIN season  s ON d.season_id  = s.season_id
JOIN weather w ON d.weather_id = w.weather_id
ORDER BY d.cnt DESC
LIMIT 3
```


**Exercise 13** — For 2011-01-01 only, return `hr`, the hourly `cnt`, and that day's daily total as `day_total`. Columns: `hr`, `cnt`, `day_total`. Order by `hr`.

```sql
SELECT h.hr, h.cnt, d.cnt AS day_total
FROM rides_hourly h
JOIN rides_daily d ON h.dteday = d.dteday
WHERE h.dteday = '2011-01-01'
ORDER BY h.hr
```


**Exercise 14** — Prove the two ride tables agree. Return one row with `daily_total`, `hourly_total` and `difference` (daily minus hourly).

```sql
SELECT (SELECT SUM(cnt) FROM rides_daily)  AS daily_total,
       (SELECT SUM(cnt) FROM rides_hourly) AS hourly_total,
       (SELECT SUM(cnt) FROM rides_daily)
       - (SELECT SUM(cnt) FROM rides_hourly) AS difference
```


**Exercise 15** — For the first 10 days of 2011 by date, return `dteday`, `cnt`, and a day-by-day running total as `running_total`. Use ROWS, not the default frame.

```sql
SELECT dteday,
       cnt,
       SUM(cnt) OVER (ORDER BY dteday
                      ROWS BETWEEN UNBOUNDED PRECEDING
                               AND CURRENT ROW) AS running_total
FROM rides_daily
WHERE year_id = 0
ORDER BY dteday
LIMIT 10
```


**Exercise 16** — Rank the 4 seasons by total rides, most first. Return `season_id`, `total` and `rank` using RANK().

```sql
SELECT season_id,
       SUM(cnt) AS total,
       RANK() OVER (ORDER BY SUM(cnt) DESC) AS rank
FROM rides_daily
GROUP BY season_id
ORDER BY rank
```


**Exercise 17** — Return every day whose `cnt` is above the overall daily average, as `dteday` and `cnt`, busiest first, limited to 5 rows.

```sql
SELECT dteday, cnt
FROM rides_daily
WHERE cnt > (SELECT AVG(cnt) FROM rides_daily)
ORDER BY cnt DESC
LIMIT 5
```


**Exercise 18** — For each month of 2012, return `mnth`, that month's total as `total`, and the previous month's total as `prev_month` (NULL for January). Order by `mnth`.

```sql
SELECT mnth,
       SUM(cnt) AS total,
       LAG(SUM(cnt)) OVER (ORDER BY mnth) AS prev_month
FROM rides_daily
WHERE year_id = 1
GROUP BY mnth
ORDER BY mnth
```


**Exercise 19** — For each hour of the day (0-23), return `hr`, the average rides on working days as `working`, and on non-working days as `non_working`, both rounded to 1 decimal. Order by `hr`. Do it in ONE query without a join.

```sql
SELECT hr,
       ROUND(AVG(CASE WHEN workingday = 1 THEN cnt END), 1)
           AS working,
       ROUND(AVG(CASE WHEN workingday = 0 THEN cnt END), 1)
           AS non_working
FROM rides_hourly
GROUP BY hr
ORDER BY hr
```


**Exercise 20** — Find the single worst day for casual riders as a share of all riders, among days with at least 1000 total rides. Return `dteday`, `casual`, `cnt` and `casual_share` rounded to 4 decimals. One row.

```sql
SELECT dteday,
       casual,
       cnt,
       ROUND(casual * 1.0 / cnt, 4) AS casual_share
FROM rides_daily
WHERE cnt >= 1000
ORDER BY casual_share ASC
LIMIT 1
```


**Exercise 21** — Return the 5 stations furthest north (largest `lat`), with `name`, `capacity`, `lat`, and their capacity rank across ALL stations as `capacity_rank` (largest capacity = rank 1, using DENSE_RANK). Order by `lat` descending.

```sql
WITH ranked AS (
    SELECT name, capacity, lat,
           DENSE_RANK() OVER (ORDER BY capacity DESC)
               AS capacity_rank
    FROM stations
)
SELECT name, capacity, lat, capacity_rank
FROM ranked
ORDER BY lat DESC
LIMIT 5
```

---

*Data Science & AI — Module 3 Part 1 practice. Database:
`data/bikeshare.db`, built by `tools/build_practice_database.py`.
Companion to `14-sql-and-databases.ipynb`.*